In [1]:
from local_pool_price import *
import plotly.express as px
import plotly.io
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from mainnet_launch.data_fetching.get_state_by_block import _build_blocks_to_use_dont_clip


plotly.io.templates.default = None


# 100 * ((backing - spot_price) / backing)
# 100 * ((safe price - spot_price) / safe price)


def fetch_pool_local_token_prices(method="(Token-ETH) / (USDC-ETH)") -> pd.DataFrame:

    backing_calls = build_backing_calls()
    token_local_price = build_all_local_pool_prices()

    token_local_price_call_list = [t.calls for t in token_local_price]
    token_local_price_calls = [c for calls in token_local_price_call_list for c in calls]

    blocks = _build_blocks_to_use_dont_clip(
        start_block=20759360, end_block=22077110, chain=ETH_CHAIN, approx_num_blocks_per_day=2
    )
    df = get_raw_state_by_blocks([*backing_calls, *token_local_price_calls], blocks, ETH_CHAIN)
    safe_to_usdc_price_df = build_safe_to_usdc_token_price(blocks, method=method)

    df = pd.concat([df, safe_to_usdc_price_df], axis=1)

    pool_spot_price_df = _extract_pool_local_spot_price_df(df, token_local_price)
    # USDs oracle returns 0 or 1.2 at the start, ignoring those outliers
    df.loc[df.index < pd.Timestamp("2024-09-25", tz="utc"), "USDs_safe_price"] = np.nan

    return df, pool_spot_price_df


def _extract_pool_local_spot_price_df(df: pd.DataFrame, token_local_price: list[TokenLocalPoolPriceDetails]):
    pool_spot_price_df = pd.DataFrame(index=df.index)

    for t in token_local_price:
        cols = [c for c in df.columns if t.pool_name in c]

        for spot_price_column in cols:
            first_token = spot_price_column.split("_")[-3]
            second_token = spot_price_column.split("_")[-1]

            pool_spot_price_df[f"{t.pool_name}__{first_token}_spot_price"] = (
                df[spot_price_column] * df[f"{second_token}_safe_price"]
            )
    return pool_spot_price_df


from mainnet_launch.constants import WORKING_DATA_DIR


def _make_sub_line_plot_for_each_column(a_df: pd.DataFrame, title: str) -> go.Figure:
    num_cols = 3
    num_rows = 1 + (len(a_df.columns) // num_cols)
    fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=list(a_df.columns))

    # Loop over each column and add its trace to the correct subplot cell
    for i, col in enumerate(a_df.columns):
        row = i // num_cols + 1
        col_pos = i % num_cols + 1

        temp_fig = px.line(a_df, y=col)
        temp_fig.update_layout(yaxis_title="Percent")

        for trace in temp_fig.data:
            fig.add_trace(trace, row=row, col=col_pos)

    fig.update_yaxes(range=[a_df.min().min() * 1.1, a_df.max().max() * 1.1])
    fig.update_layout(
        height=200 * num_rows,
        width=500 * num_cols,
        title_text=title,
        showlegend=False,
    )
    return fig


def render_spot_discount_to_backing_plots(df: pd.DataFrame, pool_spot_price_df: pd.DataFrame):
    backing_cols = [f"{col.split('_')[-3]}_backing" for col in pool_spot_price_df.columns]
    backing_df = df[backing_cols].resample("1d").last().copy()

    for n_days in [1, 3, 7]:
        moving_average_spot_price_df = pool_spot_price_df.replace(0).resample("1d").last().rolling(n_days).mean().copy()
        percent_discount_values = 100 * (moving_average_spot_price_df.values - backing_df.values) / backing_df.values

        discount_column_names = [c.replace("_spot_price", "_discount_to_backing") for c in moving_average_spot_price_df]
        percent_discount_df = pd.DataFrame(
            percent_discount_values, columns=discount_column_names, index=backing_df.index
        )

        _make_sub_line_plot_for_each_column(
            percent_discount_df, f"{n_days} Days Moving Average Pool Spot Price Discount to Backing"
        ).show()


def render_safe_price_discount_to_backing_df(df: pd.DataFrame):
    safe_price_cols = [c for c in df.columns if "_safe_price" in c]
    backing_cols = [f"{t.split('_safe_price')[0]}_backing" for t in safe_price_cols]

    safe_price_df = df[safe_price_cols].copy()
    backing_df = df[backing_cols].resample("1d").last().copy()

    for n_days in [1, 3, 7]:
        moving_average_safe_price_df = safe_price_df.resample("1d").last().rolling(n_days).mean().copy()

        percent_discount_values = 100 * (moving_average_safe_price_df.values - backing_df.values) / backing_df.values

        discount_column_names = [c.replace("_backing", "_discount_to_backing") for c in backing_df]
        percent_discount_df = pd.DataFrame(
            percent_discount_values, columns=discount_column_names, index=backing_df.index
        )

        _make_sub_line_plot_for_each_column(
            percent_discount_df, f"{n_days} Days Moving Safe Price Discount to Backing"
        ).show()


render_safe_price_discount_to_backing_df(df)


df, pool_spot_price_df = fetch_pool_local_token_prices()

[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] sslv3 alert bad record mac (_ssl.c:2548) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] sslv3 alert bad record mac (_ssl.c:2548) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] sslv3 alert bad record mac (_ssl.c:2548) [0]


In [2]:
def _make_sub_line_plot_for_each_column(a_df: pd.DataFrame, title: str) -> go.Figure:
    num_cols = 3
    num_rows = 1 + (len(a_df.columns) // num_cols)
    fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=list(a_df.columns))

    # Loop over each column and add its trace to the correct subplot cell
    for i, col in enumerate(a_df.columns):
        row = i // num_cols + 1
        col_pos = i % num_cols + 1

        temp_fig = px.line(a_df, y=col)
        temp_fig.update_layout(yaxis_title="Percent")

        for trace in temp_fig.data:
            fig.add_trace(trace, row=row, col=col_pos)

    fig.update_yaxes(range=[a_df.min().min() * 1.1, a_df.max().max() * 1.1])
    fig.update_layout(
        height=200 * num_rows,
        width=500 * num_cols,
        title_text=title,
        showlegend=False,
    )
    return fig


def render_spot_discount_to_backing_plots(df: pd.DataFrame, pool_spot_price_df: pd.DataFrame):
    backing_cols = [f"{col.split('_')[-3]}_backing" for col in pool_spot_price_df.columns]
    backing_df = df[backing_cols].resample("1d").last().copy()

    for n_days in [1, 3, 7]:
        moving_average_spot_price_df = pool_spot_price_df.replace(0).resample("1d").last().rolling(n_days).mean().copy()
        percent_discount_values = 100 * (moving_average_spot_price_df.values - backing_df.values) / backing_df.values

        discount_column_names = [c.replace("_spot_price", "_discount_to_backing") for c in moving_average_spot_price_df]
        percent_discount_df = pd.DataFrame(
            percent_discount_values, columns=discount_column_names, index=backing_df.index
        )

        _make_sub_line_plot_for_each_column(
            percent_discount_df, f"{n_days} Days Moving Average Pool Spot Price Discount to Backing"
        ).show()


def render_safe_price_discount_to_backing_df(df: pd.DataFrame):
    safe_price_cols = [c for c in df.columns if "_safe_price" in c]
    backing_cols = [f"{t.split('_safe_price')[0]}_backing" for t in safe_price_cols]

    safe_price_df = df[safe_price_cols].copy()
    backing_df = df[backing_cols].resample("1d").last().copy()

    for n_days in [1, 3, 7]:
        moving_average_safe_price_df = safe_price_df.resample("1d").last().rolling(n_days).mean().copy()

        percent_discount_values = 100 * (moving_average_safe_price_df.values - backing_df.values) / backing_df.values

        discount_column_names = [c.replace("_backing", "_discount_to_backing") for c in backing_df]
        percent_discount_df = pd.DataFrame(
            percent_discount_values, columns=discount_column_names, index=backing_df.index
        )

        _make_sub_line_plot_for_each_column(
            percent_discount_df, f"{n_days} Days Moving Safe Price Discount to Backing"
        ).show()


render_safe_price_discount_to_backing_df(df)

# RootPriceOracle not correct

In [98]:
from mainnet_launch.constants import *
from mainnet_launch.data_fetching.get_state_by_block import *
import numpy as np
from local_pool_price import *


def build_get_token_price_in_quote(token_address: str, name: str) -> Call:
    return Call(
        ROOT_PRICE_ORACLE(ETH_CHAIN),
        ["getPriceInQuote(address,address)(uint256)", token_address, USDC],
        [(name, safe_normalize_6_with_bool_success)],
    )


safe_price_calls = build_safe_price_calls()

get_token_price_in_USDC_calls = [
    build_get_token_price_in_quote(c[0], c[1] + "_to_USDC_via_rootPriceOracle") for c in stable_coin_tuples
]

blocks = [b for b in range(22020453, 22090453, 100)]
root_price_oracle_prices = get_raw_state_by_blocks(get_token_price_in_USDC_calls, blocks, ETH_CHAIN)

chainlink_usd_prices = get_raw_state_by_blocks(safe_price_calls, blocks, ETH_CHAIN)
root_price_oracle_prices["USDe_to_USDC_via_rootPriceOracle"] = np.nan
# safe_price_via_ETH_route = build_safe_to_usdc_token_price(blocks, method="(Token-ETH) / (USDC-ETH)")
safe_price_via_USDC_route = build_safe_to_usdc_token_price(blocks, method="(Token-USD) / (USDC-USD)")

In [99]:
# RootPriceOracle - Chainlink / Chainlink
root_price_cols = [
    "DAI_to_USDC_via_rootPriceOracle",
    "USDC_to_USDC_via_rootPriceOracle",
    "USDT_to_USDC_via_rootPriceOracle",
    "GHO_to_USDC_via_rootPriceOracle",
    "USDs_to_USDC_via_rootPriceOracle",
    # "USDe_to_USDC_via_rootPriceOracle",
    "FRAX_to_USDC_via_rootPriceOracle",
    "crvUSD_to_USDC_via_rootPriceOracle",
]

chainlink_price_cols = [
    "DAI_to_USD_safe_price",
    "USDC_to_USD_safe_price",
    "USDT_to_USD_safe_price",
    "GHO_to_USD_safe_price",
    "USDs_to_USD_safe_price",
    # "USDe_to_USD_safe_price",
    "FRAX_to_USD_safe_price",
    "crvUSD_to_USD_safe_price",
]

values = (
    100
    * (
        root_price_oracle_prices[root_price_cols].astype(float).values
        - chainlink_usd_prices[chainlink_price_cols].astype(float).values
    )
    / chainlink_usd_prices[chainlink_price_cols].astype(float).values
)

difference_df = pd.DataFrame(values, index=chainlink_usd_prices.index)
difference_df.columns = ["DAI", "USDC", "USDT", "GHO", "USDs", "FRAX", "crvUSD"]

fig = px.line(
    difference_df,
    labels={"y": "percent difference between Root and chainlink"},
    title="100 * Root - Chainlink / Chainlink",
)
fig.show()

In [102]:
px.line(chainlink_usd_prices["USDT_to_USD_safe_price"].astype(float), title="Chainlink USDT -> USD price")

In [107]:
px.line(
    root_price_oracle_prices["USDT_to_USDC_via_rootPriceOracle"].astype(float),
    title="Root Price Oracle USDT to USDC price",
)

In [111]:
px.line(safe_price_via_USDC_route["USDT_safe_price"], title="USDT -> USDC price via chainlink USDC route")

In [108]:
safe_price_via_USDC_route

,DAI_safe_price,USDT_safe_price,USDe_safe_price,GHO_safe_price,crvUSD_safe_price,USDs_safe_price,FRAX_safe_price,sUSDe_safe_price,USDC_safe_price,aGHO_safe_price,aUSDC_safe_price,aUSDT_safe_price
timestamp,,,,,,,,,,,,
2025-03-11 01:49:11+00:00,1.000168,1.000181,0.999851,1.000118,0.999658,1.000116,0.997623,1.158010,1.0,1.002208,1.123190,1.117795
2025-03-11 02:09:11+00:00,1.000168,1.000181,0.999851,1.000118,0.999658,1.000116,0.998083,1.158010,1.0,1.002209,1.123191,1.117796
2025-03-11 02:29:11+00:00,1.000168,1.000181,0.999851,1.000118,0.999658,1.000116,0.998083,1.158010,1.0,1.002210,1.123192,1.117797
2025-03-11 02:49:23+00:00,1.000066,1.000181,0.999851,1.000118,0.999658,1.000116,0.998083,1.158010,1.0,1.002210,1.123193,1.117798
2025-03-11 03:09:23+00:00,1.000066,1.000181,0.999851,1.000118,0.999658,1.000116,0.998245,1.158010,1.0,1.002211,1.123194,1.117800
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-19 22:39:11+00:00,1.000162,1.000073,0.999867,0.999873,0.999984,1.000044,0.998918,1.159184,1.0,1.002517,1.123923,1.118431
2025-03-19 22:59:11+00:00,1.000162,1.000073,0.999867,1.000073,1.000180,1.000129,0.998918,1.159507,1.0,1.002719,1.123924,1.118432
2025-03-19 23:19:23+00:00,1.000162,1.000073,0.999867,1.000073,1.000180,1.000129,0.999162,1.159507,1.0,1.002720,1.123925,1.118434


In [ ]:
px.line(
    root_price_oracle_prices["USDT_to_USDC_via_rootPriceOracle"].astype(float),
    title="Root Price Oracle USDT to USDC price",
)

In [75]:
difference_df.dropna().abs().mean().mean()

np.float64(0.333949449638338)

In [79]:
safe_price_via_USDC_route

,DAI_safe_price,USDT_safe_price,USDe_safe_price,GHO_safe_price,crvUSD_safe_price,USDs_safe_price,FRAX_safe_price,sUSDe_safe_price,USDC_safe_price,aGHO_safe_price,aUSDC_safe_price,aUSDT_safe_price
timestamp,,,,,,,,,,,,
2025-03-11 01:49:11+00:00,1.000168,1.000181,0.999851,1.000118,0.999658,1.000116,0.997623,1.158010,1.0,1.002208,1.123190,1.117795
2025-03-11 02:09:11+00:00,1.000168,1.000181,0.999851,1.000118,0.999658,1.000116,0.998083,1.158010,1.0,1.002209,1.123191,1.117796
2025-03-11 02:29:11+00:00,1.000168,1.000181,0.999851,1.000118,0.999658,1.000116,0.998083,1.158010,1.0,1.002210,1.123192,1.117797
2025-03-11 02:49:23+00:00,1.000066,1.000181,0.999851,1.000118,0.999658,1.000116,0.998083,1.158010,1.0,1.002210,1.123193,1.117798
2025-03-11 03:09:23+00:00,1.000066,1.000181,0.999851,1.000118,0.999658,1.000116,0.998245,1.158010,1.0,1.002211,1.123194,1.117800
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-19 22:39:11+00:00,1.000162,1.000073,0.999867,0.999873,0.999984,1.000044,0.998918,1.159184,1.0,1.002517,1.123923,1.118431
2025-03-19 22:59:11+00:00,1.000162,1.000073,0.999867,1.000073,1.000180,1.000129,0.998918,1.159507,1.0,1.002719,1.123924,1.118432
2025-03-19 23:19:23+00:00,1.000162,1.000073,0.999867,1.000073,1.000180,1.000129,0.999162,1.159507,1.0,1.002720,1.123925,1.118434


In [82]:
safe_price_via_USDC_route.columns

Index(['DAI_safe_price', 'USDT_safe_price', 'USDe_safe_price',
       'GHO_safe_price', 'crvUSD_safe_price', 'USDs_safe_price',
       'FRAX_safe_price', 'sUSDe_safe_price', 'USDC_safe_price',
       'aGHO_safe_price', 'aUSDC_safe_price', 'aUSDT_safe_price'],
      dtype='object')

In [85]:
safe_price_via_USDC_route["USDC_safe_price"] = 1

In [97]:
px.line(safe_price_via_USDC_route[usdc_route_columns])

In [96]:
px.line(safe_vs_spot_price_oracle_df[chainlink_price_cols])

In [93]:
# RootPriceOracle - Chainlink / Chainlink
root_price_cols = [
    "DAI_to_USDC_via_rootPriceOracle",
    "USDC_to_USDC_via_rootPriceOracle",
    "USDT_to_USDC_via_rootPriceOracle",
    "GHO_to_USDC_via_rootPriceOracle",
    "USDs_to_USDC_via_rootPriceOracle",
    # "USDe_to_USDC_via_rootPriceOracle",
    "FRAX_to_USDC_via_rootPriceOracle",
    "crvUSD_to_USDC_via_rootPriceOracle",
]

usdc_route_columns = [
    "DAI_safe_price",
    "USDC_safe_price",
    "USDT_safe_price",
    "GHO_safe_price",
    "USDs_safe_price",
    "FRAX_safe_price",
    "crvUSD_safe_price",
]

values = (
    100
    * (
        safe_price_via_USDC_route[usdc_route_columns].astype(float).values
        - safe_vs_spot_price_oracle_df[chainlink_price_cols].astype(float).values
    )
    / safe_vs_spot_price_oracle_df[chainlink_price_cols].astype(float).values
)

difference_df = pd.DataFrame(values, index=safe_price_via_USDC_route.index)
difference_df.columns = ["DAI", "USDC", "USDT", "GHO", "USDs", "FRAX", "crvUSD"]

fig = px.line(
    difference_df,
    labels={"y": "percent difference between Root and chainlink"},
    title="100 * (usdc route) - Chainlink / Chainlink",
)
fig.show()

In [61]:
get_state_by_one_block(get_token_price_in_USDC_calls, ETH_CHAIN.client.eth.block_number, ETH_CHAIN)

{'DAI_to_USDC_via_rootPriceOracle': 1.007374,
 'USDC_to_USDC_via_rootPriceOracle': 1.0,
 'USDT_to_USDC_via_rootPriceOracle': 1.008647,
 'GHO_to_USDC_via_rootPriceOracle': 1.002724,
 'USDs_to_USDC_via_rootPriceOracle': 1.00278,
 'USDe_to_USDC_via_rootPriceOracle': None,
 'FRAX_to_USDC_via_rootPriceOracle': 1.000963,
 'crvUSD_to_USDC_via_rootPriceOracle': 1.002832,
 'scrvUSD_to_USDC_via_rootPriceOracle': 1.044979,
 'sUSDs_to_USDC_via_rootPriceOracle': 1.047154,
 'sUSDe_to_USDC_via_rootPriceOracle': 1.162581,
 'sFRAX_to_USDC_via_rootPriceOracle': 1.121183,
 'aUSDT_to_USDC_via_rootPriceOracle': 1.128128,
 'aUSDC_to_USDC_via_rootPriceOracle': 1.123994,
 'aGHO_to_USDC_via_rootPriceOracle': 1.005435}

In [29]:
safe_vs_spot_price_oracle_df.columns

Index(['DAI_to_USD_safe_price', 'USDe_to_USD_safe_price',
       'USDC_to_USD_safe_price', 'USDT_to_USD_safe_price',
       'GHO_to_USD_safe_price', 'crvUSD_to_USD_safe_price',
       'USDs_to_USD_safe_price', 'FRAX_to_USD_safe_price',
       'sUSDe_to_USD_safe_price', 'USDC_to_ETH_safe_price',
       'USDT_to_ETH_safe_price', 'DAI_to_ETH_safe_price',
       'DAI_to_USDC_via_rootPriceOracle', 'USDC_to_USDC_via_rootPriceOracle',
       'USDT_to_USDC_via_rootPriceOracle', 'GHO_to_USDC_via_rootPriceOracle',
       'USDs_to_USDC_via_rootPriceOracle', 'USDe_to_USDC_via_rootPriceOracle',
       'FRAX_to_USDC_via_rootPriceOracle',
       'crvUSD_to_USDC_via_rootPriceOracle',
       'scrvUSD_to_USDC_via_rootPriceOracle',
       'sUSDs_to_USDC_via_rootPriceOracle',
       'sUSDe_to_USDC_via_rootPriceOracle',
       'sFRAX_to_USDC_via_rootPriceOracle',
       'aUSDT_to_USDC_via_rootPriceOracle',
       'aUSDC_to_USDC_via_rootPriceOracle',
       'aGHO_to_USDC_via_rootPriceOracle'],
      dtype='

In [32]:
dai_to_usd = pd.DataFrame(index=safe_price_via_USDC_route.index)
dai_to_usd["dai_via_usdc_path"] = safe_price_via_USDC_route["DAI_safe_price"]
dai_to_usd["DAI_to_USDC_via_rootPriceOracle"] = safe_vs_spot_price_oracle_df["DAI_to_USDC_via_rootPriceOracle"]
dai_to_usd["dai_via_usd_chainlink"] = safe_vs_spot_price_oracle_df["DAI_to_USD_safe_price"]

px.line(dai_to_usd, title="DAI to USD or USDC ")

In [37]:
gho_to_USD = pd.DataFrame(index=safe_price_via_USDC_route.index)
gho_to_USD["GHO_via_usdc_path"] = safe_price_via_USDC_route["GHO_safe_price"]
gho_to_USD["GHO_to_USDC_via_rootPriceOracle"] = safe_vs_spot_price_oracle_df["GHO_to_USDC_via_rootPriceOracle"]
gho_to_USD["GHO_to_USD_via chainlink"] = safe_vs_spot_price_oracle_df["GHO_to_USD_safe_price"]

px.line(gho_to_USD, title="GHO to USD")

In [43]:
root_price_oracle_columns = [c for c in safe_vs_spot_price_oracle_df.columns if "rootPriceOracle" in c]

px.line(safe_vs_spot_price_oracle_df[root_price_oracle_columns].astype(float))

In [49]:
root_price_oracle_columns

['DAI_to_USDC_via_rootPriceOracle',
 'USDC_to_USDC_via_rootPriceOracle',
 'USDT_to_USDC_via_rootPriceOracle',
 'GHO_to_USDC_via_rootPriceOracle',
 'USDs_to_USDC_via_rootPriceOracle',
 'USDe_to_USDC_via_rootPriceOracle',
 'FRAX_to_USDC_via_rootPriceOracle',
 'crvUSD_to_USDC_via_rootPriceOracle',
 'scrvUSD_to_USDC_via_rootPriceOracle',
 'sUSDs_to_USDC_via_rootPriceOracle',
 'sUSDe_to_USDC_via_rootPriceOracle',
 'sFRAX_to_USDC_via_rootPriceOracle',
 'aUSDT_to_USDC_via_rootPriceOracle',
 'aUSDC_to_USDC_via_rootPriceOracle',
 'aGHO_to_USDC_via_rootPriceOracle']

In [56]:
chainlink_safe_price_columns = [c for c in safe_vs_spot_price_oracle_df.columns if "_to_USD_safe_price" in c]

px.line(
    safe_vs_spot_price_oracle_df[
        [
            "DAI_to_USD_safe_price",
            "USDC_to_USD_safe_price",
            "USDT_to_USD_safe_price",
            "GHO_to_USD_safe_price",
            "USDs_to_USD_safe_price",
            "USDe_to_USD_safe_price",
            "FRAX_to_USD_safe_price",
            "crvUSD_to_USD_safe_price",
        ]
    ].astype(float),
    title="USD prices according to Chainlink (Token-USD) Oracle",
)

In [55]:
px.line(
    safe_vs_spot_price_oracle_df[
        [
            "DAI_to_USDC_via_rootPriceOracle",
            "USDC_to_USDC_via_rootPriceOracle",
            "USDT_to_USDC_via_rootPriceOracle",
            "GHO_to_USDC_via_rootPriceOracle",
            "USDs_to_USDC_via_rootPriceOracle",
            "USDe_to_USDC_via_rootPriceOracle",
            "FRAX_to_USDC_via_rootPriceOracle",
            "crvUSD_to_USDC_via_rootPriceOracle",
        ]
    ].astype(float),
    title="USDC prices according to RootPriceOracle",
)

,0,1,2,3,4,5,6,7
timestamp,,,,,,,,
2025-03-11 01:49:11+00:00,NaN,0.000191,NaN,NaN,NaN,NaN,NaN,NaN
2025-03-11 02:09:11+00:00,NaN,0.000191,NaN,NaN,NaN,NaN,NaN,NaN
2025-03-11 02:29:11+00:00,NaN,0.000191,NaN,NaN,NaN,NaN,NaN,NaN
2025-03-11 02:49:23+00:00,NaN,0.000191,NaN,NaN,NaN,NaN,NaN,NaN
2025-03-11 03:09:23+00:00,NaN,0.000191,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2025-03-19 22:39:11+00:00,0.000787,0.000073,0.001813,-0.002813,-0.002813,NaN,0.005190,-0.002813
2025-03-19 22:59:11+00:00,0.000324,0.000073,0.001233,0.001512,0.001512,NaN,0.015710,0.001512
2025-03-19 23:19:23+00:00,0.000324,0.000073,0.001233,0.001512,0.001512,NaN,0.015462,0.001512


In [46]:
chainlink_safe_price_columns

['DAI_to_USD_safe_price',
 'USDe_to_USD_safe_price',
 'USDC_to_USD_safe_price',
 'USDT_to_USD_safe_price',
 'GHO_to_USD_safe_price',
 'crvUSD_to_USD_safe_price',
 'USDs_to_USD_safe_price',
 'FRAX_to_USD_safe_price',
 'sUSDe_to_USD_safe_price']

In [44]:
safe_vs_spot_price_oracle_df.columns

Index(['DAI_to_USD_safe_price', 'USDe_to_USD_safe_price',
       'USDC_to_USD_safe_price', 'USDT_to_USD_safe_price',
       'GHO_to_USD_safe_price', 'crvUSD_to_USD_safe_price',
       'USDs_to_USD_safe_price', 'FRAX_to_USD_safe_price',
       'sUSDe_to_USD_safe_price', 'USDC_to_ETH_safe_price',
       'USDT_to_ETH_safe_price', 'DAI_to_ETH_safe_price',
       'DAI_to_USDC_via_rootPriceOracle', 'USDC_to_USDC_via_rootPriceOracle',
       'USDT_to_USDC_via_rootPriceOracle', 'GHO_to_USDC_via_rootPriceOracle',
       'USDs_to_USDC_via_rootPriceOracle', 'USDe_to_USDC_via_rootPriceOracle',
       'FRAX_to_USDC_via_rootPriceOracle',
       'crvUSD_to_USDC_via_rootPriceOracle',
       'scrvUSD_to_USDC_via_rootPriceOracle',
       'sUSDs_to_USDC_via_rootPriceOracle',
       'sUSDe_to_USDC_via_rootPriceOracle',
       'sFRAX_to_USDC_via_rootPriceOracle',
       'aUSDT_to_USDC_via_rootPriceOracle',
       'aUSDC_to_USDC_via_rootPriceOracle',
       'aGHO_to_USDC_via_rootPriceOracle'],
      dtype='

In [ ]:
root_price_oracle_columns = [c for c in safe_vs_spot_price_oracle_df.columns if "rootPriceOracle" in c]

px.line(safe_vs_spot_price_oracle_df[root_price_oracle_columns].astype(float))

In [35]:
(dai_to_usd["dai_via_usdc_path"] - dai_to_usd["DAI_to_USDC_via_rootPriceOracle"]).abs().dropna().mean()

np.float64(0.003469451060067331)

In [19]:
safe_vs_spot_price_oracle_df.columns

Index(['DAI_to_USD_safe_price', 'USDe_to_USD_safe_price',
       'USDC_to_USD_safe_price', 'USDT_to_USD_safe_price',
       'GHO_to_USD_safe_price', 'crvUSD_to_USD_safe_price',
       'USDs_to_USD_safe_price', 'FRAX_to_USD_safe_price',
       'sUSDe_to_USD_safe_price', 'USDC_to_ETH_safe_price',
       'USDT_to_ETH_safe_price', 'DAI_to_ETH_safe_price',
       'DAI_to_USDC_via_rootPriceOracle', 'USDC_to_USDC_via_rootPriceOracle',
       'USDT_to_USDC_via_rootPriceOracle', 'GHO_to_USDC_via_rootPriceOracle',
       'USDs_to_USDC_via_rootPriceOracle', 'USDe_to_USDC_via_rootPriceOracle',
       'FRAX_to_USDC_via_rootPriceOracle',
       'crvUSD_to_USDC_via_rootPriceOracle',
       'scrvUSD_to_USDC_via_rootPriceOracle',
       'sUSDs_to_USDC_via_rootPriceOracle',
       'sUSDe_to_USDC_via_rootPriceOracle',
       'sFRAX_to_USDC_via_rootPriceOracle',
       'aUSDT_to_USDC_via_rootPriceOracle',
       'aUSDC_to_USDC_via_rootPriceOracle', 'aGHO_to_USDC_via_rootPriceOracle',
       'safe_normaliz

In [21]:
import plotly.io

plotly.io.templates.default = None
px.line(safe_vs_spot_price_oracle_df[["DAI_to_USD_safe_price", "DAI_to_USDC_via_rootPriceOracle"]], "DAI USD")

In [16]:
safe_vs_spot_price_oracle_df.dtypes

DAI_to_USD_safe_price                  float64
USDe_to_USD_safe_price                 float64
USDC_to_USD_safe_price                 float64
USDT_to_USD_safe_price                 float64
GHO_to_USD_safe_price                  float64
crvUSD_to_USD_safe_price               float64
USDs_to_USD_safe_price                 float64
FRAX_to_USD_safe_price                 float64
sUSDe_to_USD_safe_price                float64
USDC_to_ETH_safe_price                 float64
USDT_to_ETH_safe_price                 float64
DAI_to_ETH_safe_price                  float64
DAI_to_USDC_via_rootPriceOracle        float64
USDC_to_USDC_via_rootPriceOracle       float64
USDT_to_USDC_via_rootPriceOracle       float64
GHO_to_USDC_via_rootPriceOracle        float64
USDs_to_USDC_via_rootPriceOracle       float64
USDe_to_USDC_via_rootPriceOracle        object
FRAX_to_USDC_via_rootPriceOracle       float64
crvUSD_to_USDC_via_rootPriceOracle     float64
scrvUSD_to_USDC_via_rootPriceOracle    float64
sUSDs_to_USDC

In [9]:
import plotly.express as px

px.line(safe_vs_spot_price_oracle_df)

ValueError: Plotly Express cannot process wide-form data with columns of different type.

In [4]:
def render_spot_discount_to_backing_plots(df: pd.DataFrame, pool_spot_price_df: pd.DataFrame):
    # Compute backing columns based on pool_spot_price_df's column naming convention
    backing_cols = [f"{col.split('_')[-3]}_backing" for col in pool_spot_price_df.columns]

    for days in [1, 3, 7]:
        # Resample and compute a rolling mean on the pool spot price data,
        # then force consolidation with .copy()
        moving_average_spot_price_df = pool_spot_price_df.resample("1d").last().rolling(days).mean().copy()

        # Resample the backing dataframe and consolidate it as well
        backing_df = df[backing_cols].resample("1d").last().copy()

        # Calculate the percent discount using element-wise DataFrame arithmetic.
        # This avoids potential alignment issues and the need to use .values.
        percent_discount_to_peg = 100 * (backing_df - moving_average_spot_price_df) / backing_df

        # Ensure we have a DataFrame with the proper index and columns
        percent_discount_to_peg_df = pd.DataFrame(
            percent_discount_to_peg,
            index=moving_average_spot_price_df.index,
            columns=moving_average_spot_price_df.columns,
        )

        print(percent_discount_to_peg_df.head())
        fig = _make_sub_line_plot_for_each_column(
            percent_discount_to_peg_df, title=f"{days} days End of Day Spot Price Discount to Backing"
        )
        fig.show()


render_spot_discount_to_backing_plots(df, pool_spot_price_df)


# def render_spot_discount_to_backing_plots(df: pd.DataFrame, pool_spot_price_df: pd.DataFrame):

#     backing_cols = [f"{col.split('_')[-3]}_backing" for col in pool_spot_price_df.columns]

#     for days in [1, 3, 7]:
#         moving_average_spot_price_df = pool_spot_price_df.resample("1d").last().rolling(days).mean().copy()
#         backing_df = df[backing_cols].resample("1d").last()
#         percent_discount_to_peg = 100 * (backing_df.values - moving_average_spot_price_df.values).div(backing_df.values)
#         percent_discount_to_peg_df = pd.DataFrame(percent_discount_to_peg, columns=moving_average_spot_price_df.columns, index= moving_average_spot_price_df.index)
#         print(percent_discount_to_peg_df.head())
#         fig = _make_sub_line_plot_for_each_column(
#             percent_discount_to_peg_df, title=f"{days} days End of Day Spot Price Discount to Backing"
#         )
#         fig.show()

# render_spot_discount_to_backing_plots(df, pool_spot_price_df)

ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
# Separate plots for each stable showing x-axis time, y axis (peg and 1 day MA safe price)
# Separate plots for each stable showing x-axis time, y axis (peg and 1 day MA spot price)
# Separate plots for each stable showing x-axis time, y axis (peg and 3 day MA safe price)
# Separate plots for each stable showing x-axis time, y axis (peg and 3 day MA spot price)
# Separate plots for each stable showing x-axis time, y axis (peg and 7 day MA safe price)
# Separate plots for each stable showing x-axis time, y axis (peg and 7 day MA spot price)



def render_by_destination_safe_and_spot_price_charts(
    df,
    pool_spot_price_df,
):

    spot_price_column_to_backing_and_safe = [(col, f"{col.split('_')[-3]}_backing", f"{col.split('_')[-3]}_safe_price" ) for col in pool_spot_price_df]

    spot_price_cols = [c[0] for c in spot_price_column_to_backing_and_safe]
    safe_price_cols = [c[2] for c in spot_price_column_to_backing_and_safe]

    backing_cols = [c[1] for c in spot_price_column_to_backing_and_safe]


    for days in [1,3,7]:
        for name, discount_to_cols in zip(['Spot Discount to Backing', 'Safe Discount to backing'], [backing_cols,safe_price_cols ]):

            moving_average_spot_price_df = pool_spot_price_df[spot_price_cols].resample("1d").last().rolling(days).average()
            reference_price_df = df[discount_to_cols].resample("1d").last()
            # backing - spot / backing
            percent_discount_to_peg = 100 * (reference_price_df - moving_average_spot_price_df).div(reference_price_df)
            fig = _make_sub_line_plot_for_each_column(percent_discount_to_peg, title=f'{days} days {name}')


            for days in [1, 3, 7]:
                


## Prob of Safe Price Premium of Discount to Peg, Spot Price Premium of Discount to Peg, Safe Spot Spread Direction

In [ ]:
def make_percent_of_time_spread_was_greater_or_less_than(spread_df: pd.DataFrame, percent_values: list[float]):
    def build_percent_of_values_less_than(x):
        def my_function(column):
            return pd.Series(100 * sum([c < x for c in column]) / len(column))

        return my_function

    def build_percent_of_values_greater_than(x):
        def my_function(column):
            return pd.Series(100 * sum([c > x for c in column]) / len(column))

        return my_function

    percent_less_than_df = pd.DataFrame()

    for p in percent_values:
        # double negative
        percent_less_than_df[f"Premium > {p}%"] = spread_df.apply(build_percent_of_values_less_than(-p)).T

    percent_greater_than_df = pd.DataFrame()
    for p in percent_values:
        percent_greater_than_df[f"Discount > {p}%"] = spread_df.apply(build_percent_of_values_greater_than(p)).T

    return pd.concat([percent_less_than_df, percent_greater_than_df], axis=1).round()


# For safe-spot spread again lets use past 6 months of data, lets use safe price in the denominator.
# In a table, note the % or probability that discount > 3, 5, 10, 20+ bps. Similarly, for premiums. We should include all stables including USDs.
# spot - safe / safe
prob_spot_price_vs_safe_price = make_percent_of_time_spread_was_greater_or_less_than(
    pool_spot_price_spread_with_safe_price_df, percent_values=[0.03, 0.05, 0.1, 0.2, 0.5]
)
prob_spot_price_vs_safe_price

In [ ]:
prob_spot_price_discount = make_percent_of_time_spread_was_greater_or_less_than(
    pool_spot_price_discount_to_backing_df, percent_values=[0, 0.1, 0.2, 0.5, 1]
)
prob_spot_price_discount

In [ ]:
prob_safe_price_discount = make_percent_of_time_spread_was_greater_or_less_than(
    safe_backing_spread_df, percent_values=[0, 0.1, 0.2, 0.5, 1]
)
prob_safe_price_discount

In [ ]:
render_n_day_subplots(
    pool_spot_price_discount_to_backing_df,
    "Spot Percent Discount to backing (Token-ETH) / (USDC-ETH) Chainlink price route",
)

In [ ]:
# safe_to_usdc_price_df = pd.DataFrame(index=raw_df.index)

# safe_to_usdc_price_df["DAI_to_USD_safe_price_via DIA->USD"] = raw_df["DAI_to_USD_safe_price"]
# safe_to_usdc_price_df["DAI_to_USDC_safe_price_via  DAI->ETH->USDC"] = (
#     raw_df["DAI_to_ETH_safe_price"] / raw_df["USDC_to_ETH_safe_price"]
# )
# safe_to_usdc_price_df["DAI_to_USDC_safe_price_via  DAI->USD->USDC"] = (
#     raw_df["DAI_to_USD_safe_price"] / raw_df["USDC_to_USD_safe_price"]
# )
# safe_to_usdc_price_df["block"] = blocks
# px.line(safe_to_usdc_price_df)

In [ ]:
def render_n_day_subplots(price_df: pd.DataFrame, name: str):

    for n_days in [1, 3, 7]:
        moving_average_df = price_df.resample("1D").last().rolling(n_days).mean()

        # Set the grid dimensions
        num_cols = 3
        num_rows = 1 + (len(price_df.columns) // num_cols)
        # Create the subplots grid with subplot titles using the column names
        fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=list(moving_average_df.columns))

        # Loop over each column and add its trace to the correct subplot cell
        for i, col in enumerate(moving_average_df.columns):
            # Calculate grid position: row and column
            row = i // num_cols + 1
            col_pos = i % num_cols + 1

            # Create a line plot for the column using px.line
            temp_fig = px.line(moving_average_df, y=col)
            temp_fig.update_layout(yaxis_title="Percent")

            # Add the traces from the temporary figure into the subplot cell
            for trace in temp_fig.data:
                fig.add_trace(trace, row=row, col=col_pos)

        fig.update_yaxes(range=[moving_average_df.min().min() * 1.1, moving_average_df.max().max() * 1.1])
        fig.update_layout(
            height=200 * num_rows,
            width=500 * num_cols,
            title_text=f"{n_days} Day Moving Average {name}",
            showlegend=False,
        )
        return